# Main model on Colab GPU (Multi-Scale PatchTST, with and without news)

Before running: **Runtime → Change runtime type → T4 GPU**.

Inputs: the GitHub repo plus two raw price files that are not in git. Upload them once to Google Drive at
`MyDrive/SeniorProject/prices/` (from `data/raw/prices/siamchart/set50_union_2019_2025.csv` and `data/raw/prices/yahoo/yahoo_events_2019_2025.csv` on the PC).

Everything is scored on the validation block only; the test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PRICE_DIR = '/content/drive/MyDrive/SeniorProject/prices'
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/

In [ ]:
# The files must be byte-identical to the PC copies (hashes recorded 2026-10-15)
import hashlib
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': 'c5c111f621ed29a9905ffa644cf761715cb3c4b8e8e8735897ae73cc2a56c0f6'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# 1) Main model WITH news (v1): about 10-30 min on a T4
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv --out data/derived/main_model_v1_news.json

In [ ]:
# 2) Main model WITHOUT news, rerun on the same GPU so the comparison is like for like
!python scripts/baselines_deep.py --models multiscale_sector --out data/derived/main_model_v0_gpu.json

In [ ]:
# 3) Optional: the multi-scale ablation partner (one scale, same sector label), with news
!python scripts/baselines_deep.py --models singlescale_sector --news data/derived/sentiment_scores.csv --out data/derived/singlescale_v1_news.json

In [ ]:
# Save results to Drive and download them; put them in data/derived/ on the PC
!mkdir -p /content/drive/MyDrive/SeniorProject/results
!cp data/derived/main_model_v1_news.json data/derived/main_model_v0_gpu.json /content/drive/MyDrive/SeniorProject/results/ 2>/dev/null
!cp data/derived/singlescale_v1_news.json /content/drive/MyDrive/SeniorProject/results/ 2>/dev/null
!ls -la /content/drive/MyDrive/SeniorProject/results/
from google.colab import files
import os
for f in ['main_model_v1_news.json', 'main_model_v0_gpu.json', 'singlescale_v1_news.json']:
    p = 'data/derived/' + f
    if os.path.exists(p): files.download(p)